# Blackbox角速度响应分析复核
主结论与限制见同目录 README.zh-CN.md。原始文件只读；图中角速度单位是deg/s。
以下单元已执行，读取完整计算脚本产生的结果，不改飞控或MPC参数。


In [1]:
from pathlib import Path
import json
import pandas as pd
base = Path('/home/sia/agilicious_internal-main/agi_ros2/analysis/blackbox_rate_response_20260331')
s = json.loads((base / 'summary.json').read_text())
print({k: s[k] for k in ('source', 'sha256', 'rows', 'duration_seconds', 'sample_rate_hz', 'gaps_over_2ms')})


{'source': '/home/sia/Downloads/1-测试飞行1(1).csv', 'sha256': '10c5ead80861a931df75b23974aa3957f53bdd7dce9dabaa3a90e2fea8938278', 'rows': 127357, 'duration_seconds': 128.430862, 'sample_rate_hz': 991.6308122264257, 'gaps_over_2ms': 0}


In [2]:
for axis, metrics in s['axes'].items():
    print(axis, 'PID到gyro(ms):', metrics['inner_bandpassed_derivative'])
    print(axis, '平滑前RC到gyro(ms):', metrics['rc_to_gyro_bandpassed_derivative'])


Roll PID到gyro(ms): {'windows': 44, 'lag_ms_p10_median_p90': [3.8574905187070465, 8.544212400474812, 11.89476716224057]}
Roll 平滑前RC到gyro(ms): {'windows': 43, 'lag_ms_p10_median_p90': [20.150540297429888, 24.429355336807596, 27.792504947549684]}
Pitch PID到gyro(ms): {'windows': 19, 'lag_ms_p10_median_p90': [7.17265084120567, 9.668384915995587, 15.173438972310201]}
Pitch 平滑前RC到gyro(ms): {'windows': 19, 'lag_ms_p10_median_p90': [23.158723552255637, 26.009282398530374, 30.99325759190995]}
Yaw PID到gyro(ms): {'windows': 43, 'lag_ms_p10_median_p90': [5.691985324049873, 7.42098813475329, 9.862622244180836]}
Yaw 平滑前RC到gyro(ms): {'windows': 43, 'lag_ms_p10_median_p90': [21.335049037806176, 23.925700432092256, 25.84023804874025]}


In [3]:
models = pd.read_csv(base / 'model_validation.csv')
print(models.loc[models.model.isin(['ideal', 'first_order_delay']), ['axis', 'stage', 'model', 'test_rmse_deg_s']].to_string(index=False))


 axis      stage             model  test_rmse_deg_s
 Roll      inner             ideal         3.294446
 Roll      inner first_order_delay         3.324430
 Roll rc_to_gyro             ideal         5.362102
 Roll rc_to_gyro first_order_delay         3.343460
Pitch      inner             ideal         3.406429
Pitch      inner first_order_delay         3.379982
Pitch rc_to_gyro             ideal         3.922808
Pitch rc_to_gyro first_order_delay         3.368679
  Yaw      inner             ideal         1.720743
  Yaw      inner first_order_delay         1.604124
  Yaw rc_to_gyro             ideal         2.805566
  Yaw rc_to_gyro first_order_delay         1.604772


In [4]:
print(json.loads((base / 'checks.json').read_text()))


{'synthetic_delayed_bandlimited_signal': {'known_delay_ms': 12, 'estimated_ms': 11.999922681348172, 'correlation': 1.0, 'passed': True}, 'source_unchanged': True}


![时序对比](rate_tracking.png)
![分窗延迟](lag_distribution.png)
## 完整复现
在终端运行（约十几秒，具体取决于平台）：
```bash
OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 MPLCONFIGDIR=/tmp/agi_blackbox_mpl python3 analyze.py '/home/sia/Downloads/1-测试飞行1(1).csv'
```
分窗分位不是置信区间；离线角速度预测改善不等于闭环位置精度改善；自动RC平滑使未来MSP响应需要重新测量。
